In [1]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio
from skimage.morphology import skeletonize
from scipy.interpolate import splprep, splev
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
import trimesh

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, tetra_mesh_from_stl, tetra_shell_from_two_surfaces, set_the_offset, export_centerline


pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/pixel_spacing.txt")

base = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/"

# offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/offset.txt")


in_folder = "segment_final/"    
out_folder = "spline_experiment/"

folder = "Radial-spline_meshes/"

if not os.path.isdir(base + folder):
    os.mkdir(base + folder)


## Load segments, pixel spacing etc

In [2]:

filename = "Segmentation_1.seg.nrrd"
export_name = base + folder + filename[:-8]

eyeball_id = 1
muslce_ids = [2,3,4,5]
pairs = [(0,2), (1,3)]

lense_id = 7

background_id = -1


data, header = nrrd.read(base + in_folder + filename)
print(data.shape)



id_max = numpy.max(data)

print("pixel spacing = ",pixel_spacing)
print("id = ", id_max )

print("dimensions = ", data.shape)

if pixel_spacing[2]!= pixel_spacing[1]:
    print("resampling")
    factor = pixel_spacing[2]/pixel_spacing[1]
    data = ndimage.zoom(data, [1,1,factor], order = 0)


print("dimensions = ", data.shape)

id = int(numpy.max(data))
print("id = ", id)




(110, 150, 155)
pixel spacing =  [0.5 0.5 0.5]
id =  7
dimensions =  (110, 150, 155)
dimensions =  (110, 150, 155)
id =  7


## Create initial meshes

In [3]:
muscles = []

for id in range(1,numpy.max(data)+1):
    if id != background_id:
    # for id in range(3,4):
        print("id = ", id)


        if len(numpy.where((data>id-0.5) & (data<id+0.5 ))[0])>0:

            mask = numpy.zeros(data.shape, dtype=numpy.uint8)
            mask[(data>id-0.5) & (data<id+0.5 )]=1

            print(len(numpy.where(mask==1)[0]))


            numpy.save(export_name +"_filtered_surf_id_"+str(id), mask)

            if id ==eyeball_id:

                mask = ndimage.median_filter(mask, size=1)
                mask[mask<0.5]=0
                mask[mask>=0.5]=1

                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], True )

                mask_voxels = numpy.argwhere(mask == 1)  # shape (N, 3), coordinates in (z, y, x)
                # compute mean voxel position
                offset = mask_voxels.mean(axis=0)*pixel_spacing[0]  # [z_mean, y_mean, x_mean]
                numpy.savetxt(base + folder + "offset.txt", offset)

                mask_eye = mask

            if id in muslce_ids:
                print("Muscle")

                muscles.append(mask)
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )

            if id == lense_id:
                mask = ndimage.median_filter(mask, size=1)
                mask[mask<0.5]=0
                mask[mask>=0.5]=1

                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )


id =  1
65767
Problem edges: 0
Outer and inner surfaces written.
id =  2
8436
Muscle
Problem edges: 4
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._surf_id_2.stl
id =  3
4587
Muscle
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._surf_id_3.stl
id =  4
7222
Muscle
Problem edges: 1
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._surf_id_4.stl
id =  5
6801
Muscle
Problem edges: 6
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._surf_id_5.stl
id =  6
6465
id =  7
1372
Problem edges: 4
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._sur

## Lense axis

In [4]:
mesh = trimesh.load(export_name +"_surf_id_"+str(lense_id)+".stl")
# mesh.vertices = (mesh.vertices + offset) / pixel_spacing

lense_center = mesh.vertices.mean(axis=0)

print("lense_center = ", lense_center)

X = mesh.vertices - lense_center

_, _, vh = numpy.linalg.svd(X)

axis_direction = vh[-1]  # first principal axis


def axis(t):
    return lense_center + t * axis_direction


axis_points = [axis(-20.0), axis(20.0)]

export_centerline(axis_points, 1.0 , base + folder + "lense_axis.vtp", [0,0,0])

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

target_point =  axis(-2.25)/pixel_spacing[0]

print("target_point = ", target_point)

lense_center =  [34.74798285 63.83954948 24.45071852]
target_point =  [ 69.28843289 123.27184881  49.78635298]


## Section curve

- eyeball mesh ∩ place perpendicular to lense axis
- output: section curve

In [5]:
eyeball_mesh = trimesh.load(export_name +"_surf_id_"+str(eyeball_id)+"_outer.stl")

section = eyeball_mesh.section(
    plane_origin=target_point*pixel_spacing[0],
    plane_normal=axis_direction
)

section_curve = []

for entity in section.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + folder + "section_curve_A.vtk",
    meshio.Mesh(
        points=section.vertices,
        cells=[("line", numpy.array(section_curve))]
    )
)

## Section curve B & radial expansion


In [6]:
target_point_B =  axis(-10)/pixel_spacing[0]

section_B = eyeball_mesh.section(
    plane_origin=target_point_B*pixel_spacing[0],
    plane_normal=axis_direction
)

section_curve_B = []

for entity in section_B.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve_B.append([idx[i], idx[i+1]])


# Center point in physical space 
center_physical = target_point_B * pixel_spacing[0]

original_vertices = numpy.array(section_B.vertices)

# Radial vectors from the center 
radial_vectors = original_vertices - center_physical

magnitudes = numpy.linalg.norm(radial_vectors, axis=1, keepdims=True)
normalized_vectors = radial_vectors / (magnitudes + 1e-8)



# Expand outward by xx mm 
expansion_distance = 2.1  
expanded_vertices = original_vertices + (normalized_vectors * expansion_distance)

section_B.vertices = expanded_vertices

# 6. Rebuild the section_curve line segments using the new expanded coordinates
expanded_section_curve = []
for entity in section_B.entities:
    idx = entity.points  # ordered indices within this specific entity
    
    for i in range(len(idx) - 1):
        # We map the local entity indices directly to our expanded_vertices array
        expanded_section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + folder + "section_curve_B.vtk",
    meshio.Mesh(
        points=section_B.vertices,
        cells=[("line", numpy.array(expanded_section_curve))]
    )
)

## Functions: 
### Fit centerlines with/without additional constraints

In [7]:
def fit_target_constrained_spline(binned_centers, target_point, smoothing_factor=1.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_point: (3,) array [z, y, x] where the object should end
    """
    # 1. Append the target point to the end of your centers
    # We ensure target_point is shaped correctly for concatenation
    target_point = numpy.array(target_point).reshape(1, 3)
    constrained_points = numpy.concatenate([binned_centers, target_point], axis=0)
    
    # 2. Assign Weights (w)
    # We give the target point a very high weight (e.g., 10) 
    # and the noisy data points a lower weight (e.g., 1)
    weights = numpy.ones(len(constrained_points))
    weights[-1] = 20.0  # Force the spline to prioritize hitting the target
    
    # 3. Fit the Spline
    # We use the transpose .T because splprep expects (3, N)
    try:
        # Note: 'w' is the weights, 's' is the overall smoothing
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=3)
        
        # 4. Evaluate from 0 to 1
        # Since the target is now the last point in the data, 
        # u=1.0 will correspond to the target point.
        u_new = numpy.linspace(-0.05, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Constrained spline fit failed: {e}")
        return None
    
def fit_3_constrained_spline(binned_centers, target_1, target_2, target_3, smoothing_factor=1.5):
    """
    Fits a spline through binned centers and 3 target guiding points,
    """
    # 1. Combine all points into one unordered pool
    target_1 = numpy.array(target_1).reshape(1, 3)
    target_2 = numpy.array(target_2).reshape(1, 3)
    target_3 = numpy.array(target_3).reshape(1, 3)
    
    unordered_points = numpy.concatenate([binned_centers, target_1], axis=0)
    
    # 2. Use PCA  to find the true main orientation axis
    pca = PCA(n_components=1)
    projection = pca.fit_transform(unordered_points)
    
    # 3. Sort ALL points based on their position along this principal axis
    sort_idx = numpy.argsort(projection.flatten())
    constrained_points = unordered_points[sort_idx]

    constrained_points = numpy.concatenate([constrained_points, target_2, target_3], axis=0)
    
    # 4. Flip check (Ensure Target 3 is at the end)
    distances_to_end = numpy.linalg.norm(constrained_points - target_3, axis=1)
    if numpy.argmin(distances_to_end) == 0:
        constrained_points = numpy.flip(constrained_points, axis=0)

    # 5. Assign Weights dynamically by finding where the targets landed
    weights = numpy.ones(len(constrained_points))
    
    for target in [target_1]:
        distances = numpy.linalg.norm(constrained_points - target, axis=1)
        matched_idx = numpy.argmin(distances)
        weights[matched_idx] = 50
    
    weights[0] = 20
    weights[-1] = 20

    # 6. Fit the Spline
    try:
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=3)
        
        # Evaluate slightly past the boundaries to allow mathematical extension
        u_new = numpy.linspace(-0.05, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    
    except Exception as e:
        print(f"PCA-Ordered 3-point constrained spline fit failed: {e}")
        return None
    

def fit_dual_constrained_spline(binned_centers, target_1, target_2, smoothing_factor=1.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_1: (3,) array where the object should begin
    target_2: (3,) array where the object should end
    """
    # 1. Prepare the sequence: [Start, Binned Data, End]
    points = [
        binned_centers,
        numpy.array(target_1).reshape(1, 3),
        numpy.array(target_2).reshape(1, 3)
    ]
    constrained_points = numpy.concatenate(points, axis=0)
    
    # 2. Assign Weights
    # Weights[0] is Start, Weights[-1] is End. 
    # High values force the spline to hit those exact coordinates.
    weights = numpy.ones(len(constrained_points))
    weights[0] = 50.0  # Force start
    weights[-1] = 50.0 # Force end
    
    # 3. Fit the Spline
    # We pass the weights array to 'w'
    try:
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=3)
        
        # 4. Evaluate
        # u=0.0 will be target_start, u=1.0 will be target_end
        u_new = numpy.linspace(-0.05, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Dual constrained spline fit failed: {e}")
        return None

def fit_analytical_centerline(mask, target_point, ignored_nodes, n_bins=20, extension_factor=0.2):
        # 1. Get all voxel coordinates (Z, Y, X)
        points = numpy.argwhere(mask == 1).astype(float)
        
        if len(points) < 10:
            return None

        # 2. Find the main orientation using PCA
        pca = PCA(n_components=1)
        # Project points onto the first principal component 
        projection = pca.fit_transform(points)
        
        # 3. Sort points based on their position along the principal axis
        sort_idx = numpy.argsort(projection.flatten())

        sort_idx = sort_idx[int(len(sort_idx)*ignored_nodes):int(len(sort_idx)*(1-ignored_nodes))]

        sorted_points = points[sort_idx]
        sorted_projection = projection[sort_idx].flatten()

        # 4. Create "Binned Centers" to handle the noise
        # Fit to the average center of 'n' slices
        bins = numpy.linspace(sorted_projection.min(), sorted_projection.max(), n_bins)
        binned_centers = []
        
        for i in range(len(bins)-1):
            # Find points that fall within this slice of the object
            mask_slice = (sorted_projection >= bins[i]) & (sorted_projection < bins[i+1])
            if numpy.any(mask_slice):
                center = sorted_points[mask_slice].mean(axis=0)
                binned_centers.append(center)
        
        binned_centers = numpy.array(binned_centers)

        # # 5. Fit the Spline to the centers
        # # Smoothing 's' can be lower here because the binning already cleaned the noise
        tck, u = splprep(binned_centers.T, s=3.5, k=2)
        
        # # 6. Extend the object
        # # Evaluating from 0.0 to 1+extension extends the centerline
        smooth_centerline = numpy.array(splev(numpy.linspace(-extension_factor, 1.0, 100), tck)).T

        if len(target_point)==0:
            return smooth_centerline
        
        if len(target_point)==1:
            smooth_centerline_extended = fit_target_constrained_spline(binned_centers, target_point[0])
            return smooth_centerline, smooth_centerline_extended
        
        if len(target_point)==2:
            smooth_centerline_extended = fit_dual_constrained_spline(binned_centers, target_point[0], target_point[1] )
            return smooth_centerline, smooth_centerline_extended

        if len(target_point)==3:
            smooth_centerline_extended = fit_3_constrained_spline(binned_centers, target_point[0], target_point[1], target_point[2] )
            return smooth_centerline, smooth_centerline_extended    


## Functions:
### Get insertion points ~ section curve / fitted planes

In [8]:
def fit_plane_to_two_centerlines(centerline1, centerline2):
    # 1. Combine all points from both centerlines into one cloud
    points = numpy.vstack((centerline1, centerline2))
    
    # 2. Calculate the centroid (the average point)
    centroid = numpy.mean(points, axis=0)
    
    # 3. Center the points by subtracting the centroid
    centered_points = points - centroid
    
    # 4. Use Singular Value Decomposition
    _, _, vh = numpy.linalg.svd(centered_points)
    
    # 5. The normal vector 'n' is the last row of Vh 
    # (the direction with the least variance/spread)
    normal = vh[-1]
    
    return centroid, normal/numpy.linalg.norm(normal)

def find_plane_curve_intersections(section, section_curve, plane_centroid, plane_normal):
    intersections = []
    
    # Normalize plane normal just in case
    plane_normal = plane_normal / numpy.linalg.norm(plane_normal)
    
    vertices = section.vertices
    
    for start_idx, end_idx in section_curve:
        p1 = vertices[start_idx]
        p2 = vertices[end_idx]
        
        # Signed distances to the plane
        d1 = numpy.dot(p1 - plane_centroid, plane_normal)
        d2 = numpy.dot(p2 - plane_centroid, plane_normal)
        
        # Check if the segment crosses the plane
        if d1 * d2 < 0:
            # Linear interpolation to find the exact crossing point
            t = abs(d1) / (abs(d1) + abs(d2))
            intersect_pt = p1 + t * (p2 - p1)
            intersections.append(intersect_pt)
            
    return numpy.array(intersections)



In [9]:
extended_centerlines = []
centerlines = []
extended_centerlines_1 = []

# target_point = point on the lense axis

for muscle_id in range(len(muslce_ids)):
    mask = muscles[muscle_id]
    ignored_nodes = 0.01

    centerline, centerline_ext_1 = fit_analytical_centerline(mask, [target_point], ignored_nodes, n_bins=10, extension_factor=0.15)
    
    centerlines.append(centerline)
    extended_centerlines.append(centerline_ext_1)


    export_centerline(centerline_ext_1, pixel_spacing[0], base + folder + "centerline_exttended_1_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])
    export_centerline(centerline, pixel_spacing[0], base + folder + "centerline_init_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])


point_hor, normal_vector_hor = fit_plane_to_two_centerlines(extended_centerlines[pairs[0][0]], extended_centerlines[pairs[0][1]])
point_vert, normal_vector_vert = fit_plane_to_two_centerlines(extended_centerlines[pairs[1][0]], extended_centerlines[pairs[1][1]])

print("Check of angle  = ", numpy.dot(normal_vector_hor, normal_vector_vert))

# Plane 1 intersections
pts_plane1 = find_plane_curve_intersections(section, section_curve, point_hor*pixel_spacing[0], normal_vector_hor)
# Plane 2 intersections
pts_plane2 = find_plane_curve_intersections(section, section_curve, point_vert*pixel_spacing[0], normal_vector_vert)


# Plane 1B intersections
pts_plane1_B = find_plane_curve_intersections(section_B, section_curve_B, point_hor*pixel_spacing[0], normal_vector_hor)
# Plane 2B intersections
pts_plane2_B = find_plane_curve_intersections(section_B, section_curve_B, point_vert*pixel_spacing[0], normal_vector_vert)

for muscle_id in range(len(muslce_ids)):

    if muscle_id == pairs[0][0]:
        target_1 = pts_plane1[1]/pixel_spacing[0]
        target_1_B = pts_plane1_B[1]/pixel_spacing[0]

    elif muscle_id == pairs[0][1]:
        target_1 = pts_plane1[0]/pixel_spacing[0]
        target_1_B = pts_plane1_B[0]/pixel_spacing[0]

    if muscle_id == pairs[1][1]:
        target_1 = pts_plane2[0]/pixel_spacing[0]
        target_1_B = pts_plane2_B[0]/pixel_spacing[0]

    elif muscle_id == pairs[1][0]:
        target_1 = pts_plane2[1]/pixel_spacing[0]
        target_1_B = pts_plane2_B[1]/pixel_spacing[0]

    mask = muscles[muscle_id]

    _, centerline_ext_2 = fit_analytical_centerline(mask, [target_1_B, target_1, target_point], ignored_nodes, n_bins=10, extension_factor=0.25)

    extended_centerlines_1.append(centerline_ext_2)


    export_centerline(centerline_ext_2, pixel_spacing[0], base + folder + "centerline_exttended_2_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])



Check of angle  =  0.05153277060902848


## Find intersection of the extended centerline and the eyeball


In [10]:
def find_eye_intersection_index(centerline_extended, mask_eye):
    """
    Returns the index along the extended centerline that intersects the eye.
    """
    for idx, point in enumerate(centerline_extended):
        z, y, x = numpy.round(point).astype(int)
        
        # Check bounds
        if 0 <= z < mask_eye.shape[0] and 0 <= y < mask_eye.shape[1] and 0 <= x < mask_eye.shape[2]:
            if mask_eye[z, y, x] == 1:
                return idx  
                
    return len(centerline_extended) - 1


## Functions: map radial profiles, generate muscle

In [11]:

def get_local_frame(direction):
    """Calculates two vectors perpendicular to the spine direction."""
    # Pick a non-parallel vector to start the cross product
    vec = numpy.array([1, 0, 0]) if abs(direction[0]) < 0.9 else numpy.array([0, 1, 0])
    v1 = numpy.cross(direction, vec)
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(direction, v1)
    return v1, v2

def spline_radial_profiles(centerline, centerline_extended, init_mesh_surf,  intersection_idx, n_angles=12):
    # 1. Setup and Precomputation
    # Scale mesh vertices to match voxel spacing
    mesh_points = numpy.array(init_mesh_surf.vertices / pixel_spacing[0])   
    mesh_tree = cKDTree(mesh_points)
    
    angles = numpy.linspace(0, 2 * numpy.pi, n_angles, endpoint=False)
    radial_profiles = [] 

    # 2. SAMPLING: Map the original shape to radial ribs
    for i, p in enumerate(centerline):
        direction = centerline[min(i+1, len(centerline)-1)] - centerline[max(i-1, 0)]
        direction = direction / numpy.linalg.norm(direction)
        v1, v2 = get_local_frame(direction)
        
        # 1. Start with a local neighborhood ball to get candidates
        idx = mesh_tree.query_ball_point(p, r=45.0)
        local_pts = mesh_points[idx] - p

        # 2. Project points onto the spine direction (Normal to the plane)
        dist_from_plane = numpy.dot(local_pts, direction)

        # 3. Keep only points within a thin 'slab' (e.g., 10 units thick)
        thickness = 10.0
        mask_slab = numpy.abs(dist_from_plane) < (thickness / 2.0)
        local_pts = local_pts[mask_slab]
                
        if len(local_pts) > 5:
            x_p = numpy.dot(local_pts, v1)
            y_p = numpy.dot(local_pts, v2)
            
            pt_angles = numpy.arctan2(y_p, x_p) % (2 * numpy.pi)
            pt_dists = numpy.sqrt(x_p**2 + y_p**2)
            
            current_profile = []
            for a in angles:
                angular_diff = numpy.abs(pt_angles - a)
                angular_diff = numpy.minimum(angular_diff, 2 * numpy.pi - angular_diff)
                
                mask_slice = angular_diff < (numpy.pi / n_angles)

                if numpy.any(mask_slice):
                    current_profile.append(numpy.median(pt_dists[mask_slice]))
                else:
                    current_profile.append(numpy.nan)
            radial_profiles.append(current_profile)
        else:
            radial_profiles.append([numpy.nan] * n_angles)

    # 3. Spline fit(SMOOTHING) & EXTRAPOLATION
    radial_profiles = numpy.array(radial_profiles)
    
    # define coordinate space using actual point indices
    u = numpy.arange(len(radial_profiles))
    u_full = numpy.arange(intersection_idx + 1)
    
    # output ~ full extended centerline size
    smooth_profiles = numpy.zeros((len(centerline_extended), n_angles))

    for a in range(n_angles):
        column = radial_profiles[:, a]
        valid = ~numpy.isnan(column)
        
        if numpy.sum(valid) > 4:
            # Fit the spline 
            rib_spline = UnivariateSpline(u[valid], column[valid], k=3, s=15, ext=0)
            
            # Extrapolate up to the eye intersection index
            extrapolated_radii = rib_spline(u_full)
            
            # Check for negative values
            extrapolated_radii = numpy.clip(extrapolated_radii, 0.1, None)
            
            # Paste the values into the matrix up to the intersection zone
            smooth_profiles[:intersection_idx + 1, a] = extrapolated_radii
        else:
            smooth_profiles[:intersection_idx + 1, a] = 1.0

    return smooth_profiles


def reconstruct_variable_volume(smooth_profiles, centerline_extended, original_shape):
    n_angles = smooth_profiles.shape[1]
    
    # 1. Precompute local coordinate frames for the extended spine
    v1_list = []
    v2_list = []
    for i, p in enumerate(centerline_extended):
        direction = centerline_extended[min(i+1, len(centerline_extended)-1)] - centerline_extended[max(i-1, 0)]
        direction = direction / numpy.linalg.norm(direction)
        v1, v2 = get_local_frame(direction)
        v1_list.append(v1)
        v2_list.append(v2)
        
    v1_list = numpy.array(v1_list)
    v2_list = numpy.array(v2_list)

    # 2. Setup the output mask volume canvas
    new_mask = numpy.zeros(original_shape, dtype=numpy.uint8)
    max_r = smooth_profiles.max()
    
    # Bounding box constraints
    z_min, y_min, x_min = numpy.floor(centerline_extended.min(axis=0) - max_r - 1).astype(int).clip(0)
    z_max, y_max, x_max = numpy.ceil(centerline_extended.max(axis=0) + max_r + 1).astype(int)
    z_max = min(z_max, original_shape[0])
    y_max = min(y_max, original_shape[1])
    x_max = min(x_max, original_shape[2])

    z_grid, y_grid, x_grid = numpy.ogrid[z_min:z_max, y_min:y_max, x_min:x_max]
    grid_coords = numpy.stack(numpy.meshgrid(z_grid.ravel(), y_grid.ravel(), x_grid.ravel(), indexing='ij'), axis=-1)

    # 3. Project voxel grid onto the centerline spine
    tree_spine = cKDTree(centerline_extended)
    dist_to_spine, idx_on_spine = tree_spine.query(grid_coords)

    # 4. Orientation transformation
    diffs = grid_coords - centerline_extended[idx_on_spine]
    v1_local = v1_list[idx_on_spine]
    v2_local = v2_list[idx_on_spine]

    # Convert coordinates into localized 2D cross-section variables
    dots_v1 = numpy.einsum('ij,ij->i', diffs.reshape(-1, 3), v1_local.reshape(-1, 3))
    dots_v2 = numpy.einsum('ij,ij->i', diffs.reshape(-1, 3), v2_local.reshape(-1, 3))

    voxel_angles = numpy.arctan2(dots_v2, dots_v1) % (2 * numpy.pi)
    
    # 5. Profile interpolation
    angle_idx = (voxel_angles / (2 * numpy.pi) * n_angles)
    idx_low = numpy.floor(angle_idx).astype(int) % n_angles
    idx_high = (idx_low + 1) % n_angles
    weight = angle_idx - numpy.floor(angle_idx)

    r_low = smooth_profiles[idx_on_spine.flatten(), idx_low]
    r_high = smooth_profiles[idx_on_spine.flatten(), idx_high]
    local_threshold_radii = (1 - weight) * r_low + weight * r_high

    # Create mask: True if voxel falls inside the interpolated boundary profile
    local_mask = (dist_to_spine.flatten() <= local_threshold_radii).astype(numpy.uint8)
    new_mask[z_min:z_max, y_min:y_max, x_min:x_max] = local_mask.reshape(z_grid.size, y_grid.size, x_grid.size)

    return new_mask

In [12]:
radial_profiles = []
h=1.0

for muscle_id in range(len(muslce_ids)):
    mask = muscles[muscle_id]
    init_mesh = trimesh.load(export_name +"_surf_id_"+str(muslce_ids[muscle_id])+".stl")


    # 1. Find the intersection index 
    intersection_idx = find_eye_intersection_index(extended_centerlines_1[muscle_id], mask_eye)

    # 2. Map the profiles & extrapolate
    smooth_profiles = spline_radial_profiles(
        centerlines[muscle_id], 
        extended_centerlines_1[muscle_id], 
        init_mesh,
        intersection_idx=intersection_idx 
    )

    # 3. Generate new mask
    new_mask = reconstruct_variable_volume(smooth_profiles, extended_centerlines_1[muscle_id], data.shape)
    
    new_mask[mask_eye==1]=0

    new_mask = ndimage.median_filter(new_mask, size=3)

    radial_profiles.append(smooth_profiles)



    # 4. Generate mesh based on the mask
    mesh_surf = getSurfaceMesh(new_mask, export_name +"_RADIAL_surf_id_"+str(muslce_ids[muscle_id])+".stl", pixel_spacing[0], False )



    tetra_mesh_from_stl(export_name +"_RADIAL_surf_id_"+str(muslce_ids[muscle_id])+".stl", export_name + "RAD_id_"+str(muslce_ids[muscle_id])+"_" + "_h_"+str(h).replace(".","-"), h)


Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._RADIAL_surf_id_2.stl
Info    : Reading '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._RADIAL_surf_id_2.stl'...
Info    : 11292 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._RADIAL_surf_id_2.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 11292 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 5677 triangles split in 2 parts because parametrized triangles are too small (3.00538e-09)
Info    :  - Level 1 partition with 5615 triangles split in 2 parts

Info    : 74 elements combined 
Info    : ImproveMesh 
Info    : Total badness = 11442.8 
Info    : Total badness = 10166.1 
Info    : SplitImprove 
Info    : badmax = 824.667 
Info    : 1 splits performed 
Info    : ImproveMesh 
Info    : Total badness = 10178.6 
Info    : Total badness = 10071 
Info    : SwapImprove  
Info    : 399 swaps performed 
Info    : SwapImprove2  
Info    : 0 swaps performed 
Info    : ImproveMesh 
Info    : Total badness = 9310.78 
Info    : Total badness = 8993.32 
Info    : CombineImprove 
Info    : 14 elements combined 
Info    : ImproveMesh 
Info    : Total badness = 8840.31 
Info    : Total badness = 8808.02 
Info    : SplitImprove 
Info    : badmax = 137.785 
Info    : 0 splits performed 
Info    : ImproveMesh 
Info    : Total badness = 8808.02 
Info    : Total badness = 8804.55 
Info    : SwapImprove  
Info    : 155 swaps performed 
Info    : SwapImprove2  
Info    : 0 swaps performed 
Info    : ImproveMesh 
Info    : Total badness = 8602.8 
Info    

Problem edges: 2
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._RADIAL_surf_id_3.stl
Info    : Reading '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._RADIAL_surf_id_3.stl'...
Info    : 8052 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/Segmentation_1._RADIAL_surf_id_3.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 8052 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 4055 triangles split in 2 parts because parametrized triangles are too small (1.17976e-10)
Info    :  - Level 2 partition with 2038 triangles split in 2 parts b

Error   : The 1D mesh seems not to be forming a closed loop (4 boundary nodes are considered once)


Exception: The 1D mesh seems not to be forming a closed loop (4 boundary nodes are considered once)

## Visualization

In [ ]:
# def generate_straightened_volume(smooth_profiles, max_radius=None, radial_resolution=1):
#     """
#     Generates a 3D binary volume of the object as if the centerline were a straight line.
    
#     Output Shape: (num_centerline_points, 2 * max_radius, 2 * max_radius)
#     Z-axis: Index along the centerline
#     Y, X-axes: Cross-section at that point
#     """
#     n_points, n_angles = smooth_profiles.shape
    
#     if max_radius is None:
#         # Determine canvas size based on the maximum radius found
#         max_r = int(numpy.ceil(smooth_profiles.max())) + 2
#     else:
#         max_r = max_radius

#     # 1. Create a 2D grid for the cross-section (centered at 0,0)
#     side_length = 2 * max_r
#     y, x = numpy.ogrid[-max_r:max_r, -max_r:max_r]
    
#     # Calculate the flat distance and angle for every pixel in the cross-section
#     pixel_dists = numpy.sqrt(x**2 + y**2)
#     pixel_angles = numpy.arctan2(y, x) % (2 * numpy.pi)
    
#     # Map pixel angles to profile column indices
#     angle_idx = (pixel_angles / (2 * numpy.pi) * n_angles)
#     idx_low = numpy.floor(angle_idx).astype(int) % n_angles
#     idx_high = (idx_low + 1) % n_angles
#     weight = angle_idx - numpy.floor(angle_idx)
    
#     # 2. Allocate the 3D straightened mask volume
#     straightened_mask = numpy.zeros((n_points, side_length, side_length), dtype=numpy.uint8)
    
#     # 3. Reconstruct slice-by-slice along the straight axis
#     for z in range(n_points):
#         r_low = smooth_profiles[z, idx_low]
#         r_high = smooth_profiles[z, idx_high]
        
#         # Linearly interpolate the radius limit for this specific slice's angles
#         local_threshold_radii = (1 - weight) * r_low + weight * r_high
        
#         # Voxel is filled if its local radius is within the profile threshold
#         straightened_mask[z] = (pixel_dists <= local_threshold_radii).astype(numpy.uint8)
        
#     return straightened_mask


# for muscle_id in range(len(muslce_ids)):


#     straight_mask = generate_straightened_volume(radial_profiles[muscle_id])

#     mesh_surf = getSurfaceMesh(straight_mask, base + folder  +"straight_muscle_id_"+str(muslce_ids[muscle_id])+".stl", pixel_spacing[0], False )


Problem edges: 11
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/straight_muscle_id_2.stl
Problem edges: 18
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-spline_meshes/straight_muscle_id_3.stl


IndexError: list index out of range